In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
import pickle
from tqdm import tqdm

In [2]:
DATASET_PATH = ""

TARGET_SIZE = (96, 96)
NUM_CLASSES = 8
TRAIN_RATIO = 0.7
CAL_RATIO = 0.15
TEST_RATIO = 0.15

CLASS_NAMES = {
    '00': 'palm',
    '01': '1',
    '02': 'fist',
    '03': 'thumb',
    '04': 'index',
    '05': 'ok',
    '06': 'c',
    '07': 'down'
}

In [3]:
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, TARGET_SIZE)
    normalized = resized.astype('float32') / 255.0
    return normalized

def augment_image(image):
    return image

test_img_path = os.path.join(
    DATASET_PATH,
    "00",
    "01_palm",
    "frame_00_01_0001.png"
)
if os.path.exists(test_img_path):
    test_result = preprocess_image(test_img_path)
    print("Shape of test_result:", test_result.shape)

In [ ]:
def collect_data():
    images = []
    labels = []
    gesture_folders = [
        '01_palm', '02_l', '03_fist', '05_thumb',
        '06_index', '07_ok', '09_c', '10_down'
    ]

    gesture_to_label = {
        '01_palm': 0,
        '02_l': 1,
        '03_fist': 2,
        '05_thumb': 3,
        '06_index': 4,
        '07_ok': 5,
        '09_c': 6,
        '10_down': 7
    }

    for class_dir in [f"{i:02d}" for i in range(10)]:
        class_path = os.path.join(DATASET_PATH, class_dir)

        for gesture_folder in gesture_folders:
            gesture_path = os.path.join(class_path, gesture_folder)
            if not os.path.exists(gesture_path):
                continue

            print(f"{class_dir}: {gesture_folder}")

            for img_name in tqdm(os.listdir(gesture_path)):
                if img_name.endswith('.png'):
                    img_path = os.path.join(gesture_path, img_name)
                    processed_img = preprocess_image(img_path)
                    images.append(processed_img)
                    labels.append(gesture_to_label[gesture_folder])

    return np.array(images), np.array(labels)

images, labels = collect_data()
print("\nFinish Collect data!")
print("Images Length: ", len(images))
print("Labels Length: ", len(labels))

In [ ]:
def split_and_save_data(images, labels):
    X_temp, X_test, y_temp, y_test = train_test_split(
        images,
        labels,
        test_size=TEST_RATIO,
        random_state=42,
        stratify=labels
    )

    X_train, X_cal, y_train, y_cal = train_test_split(
        X_temp,
        y_temp,
        test_size=CAL_RATIO/(TRAIN_RATIO + CAL_RATIO),
        random_state=42,
        stratify=y_temp
    )

    return {
        'train': (X_train, y_train),
        'cal': (X_cal, y_cal),
        'test': (X_test, y_test)
    }


datasets = split_and_save_data(images, labels)
print("Split and Save data Successfully!")

In [ ]:
def verify_data(name, X, y):
    print(f"\nVerify {name} data:")
    assert len(X) == len(y), f"{name} hasn't the same X length and y length!"
    print(f"X has length: {len(X)}")

    for i in range(NUM_CLASSES):
        count = np.sum(y == i)
        print(f"{i} ({CLASS_NAMES[f'{i:02d}']}): {count}")
    
    return True

for name, (X, y) in datasets.items():
    verify_data(name, X, y)

Extract .pkl files

In [ ]:
for name, (X, y) in datasets.items():
    with open(f'{name}.pkl', 'wb') as f:
        pickle.dump((X, y), f)
    print(f'\nextract {name}.pkl successfully!')

print("\n Save all .pkl successfully!")

Validate shape

In [ ]:
with open('train.pkl', 'rb') as f:
    X_train, y_train = pickle.load(f)

with open('cal.pkl', 'rb') as f:
    X_cal, y_cal = pickle.load(f)

with open('test.pkl', 'rb') as f:
    X_test, y_test = pickle.load(f)

print("X_train shape: ", X_train.shape)
print("y_train shape: ", y_train.shape)

Load sample

In [ ]:
with open('train.pkl', 'rb') as f:
    X_train, y_train = pickle.load(f)

print("Type: ", X_train.dtype) # float32
print("Range: ", X_train.min(), " - ", X_train.max()) # 0-1
print("Shape: ", X_train.shape) # (15999, 96, 96)

X_train_with_channel = X_train[..., np.newaxis]
print("X_train with channel Shape: ", X_train_with_channel.shape) # (15999, 96, 96, 1)

import matplotlib.pyplot as plt
plt.show(X_train[1500], cmap='gray')
plt.axis('off')
plt.show()